In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
PATH = "../data/internet_usage.csv"
W = 5                      
EXCLUDE = []               
TRAIN_END = 2016           
VAL_END = 2019             
TEST_END = 2022            
SEEDS = [42, 1, 2, 3, 4]
EPOCHS = 300
BATCH = 64

In [3]:
def load_panel():
    df = pd.read_csv(PATH, na_values=[".."])
    years = [c for c in df.columns if c.isdigit()]
    df = df[~df["Country Code"].isin(EXCLUDE)]
    return df, years

In [4]:
def logit(p):
    p = np.clip(p, 0.01, 99.99) / 100.0
    return np.log(p / (1 - p))


In [5]:
def inv_logit(z):
    return 100.0 / (1.0 + np.exp(-z))

In [6]:
def make_windows(df, years):
    """Her ülke-yıl için: son W yılın logit değerleri (X), hedef yılın logit değeri (Y)."""
    X, Y, meta = [], [], []
    for _, r in df.iterrows():
        pct = r[years].values.astype(float)
        z = logit(pct)
        for i in range(W, len(years)):
            t = int(years[i])
            x = z[i - W:i]
            y = z[i]
            if np.isnan(x).any() or np.isnan(y):
                continue
            X.append(x)
            Y.append(y)
            meta.append((r["Country Code"], t, pct[i]))
    X = np.array(X)
    Y = np.array(Y)
    meta = pd.DataFrame(meta, columns=["code", "target_year", "y_pct"])
    return X, Y, meta

In [7]:
def split_masks(meta):
    ty = meta["target_year"].values
    tr = ty <= TRAIN_END
    va = (ty > TRAIN_END) & (ty <= VAL_END)
    te = (ty > VAL_END) & (ty <= TEST_END)
    return tr, va, te


In [8]:
def baselines(X, meta, te):
    y_true = meta["y_pct"].values
    last = X[:, -1]
    drift = (X[:, -1] - X[:, 0]) / (W - 1)

    preds = {
        "naive": inv_logit(last),
        "drift": inv_logit(last + drift),
    }
    return {k: np.mean(np.abs(p[te] - y_true[te])) for k, p in preds.items()}

In [9]:
def build_model():
    m = keras.Sequential([
        layers.Input(shape=(W - 1, 1)),
        layers.Conv1D(16, 2, activation="relu"),
        layers.Conv1D(16, 2, activation="relu"),
        layers.Flatten(),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mae")
    return m

In [10]:
def run_cnn(X, Y, tr, va, seed):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    last = X[:, -1]
    X_rel = X[:, :-1] - last[:, None]   # son değere göre farklar
    y_rel = Y - last                    # hedef: değişim

    sd = y_rel[tr].std()                # ölçek yalnızca eğitimden
    Xs = (X_rel / sd)[..., None]
    ys = y_rel / sd

    model = build_model()
    es = keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
    model.fit(Xs[tr], ys[tr],
              validation_data=(Xs[va], ys[va]),
              epochs=EPOCHS, batch_size=BATCH,
              callbacks=[es], verbose=0)

    pred_rel = model.predict(Xs, verbose=0).ravel() * sd
    return inv_logit(last + pred_rel)


In [11]:
def mae(pred, y_true, mask):
    return np.mean(np.abs(pred[mask] - y_true[mask]))

In [12]:
def main():
    df, years = load_panel()
    X, Y, meta = make_windows(df, years)
    tr, va, te = split_masks(meta)
    y_true = meta["y_pct"].values

    print(f"pencere  train={tr.sum()}  val={va.sum()}  test={te.sum()}")

    base = baselines(X, meta, te)
    print(f"naive  test MAE: {base['naive']:.3f}")
    print(f"drift  test MAE: {base['drift']:.3f}")

    cnn_maes = []
    for seed in SEEDS:
        keras.backend.clear_session()
        pred = run_cnn(X, Y, tr, va, seed)
        m = mae(pred, y_true, te)
        cnn_maes.append(m)
        print(f"CNN seed={seed}  test MAE: {m:.3f}")

    cnn_maes = np.array(cnn_maes)
    print(f"CNN ortalama: {cnn_maes.mean():.3f}  std: {cnn_maes.std():.3f}")




In [13]:
if __name__ == "__main__":
    main()

pencere  train=2331  val=544  test=509
naive  test MAE: 3.333
drift  test MAE: 2.375

CNN seed=42  test MAE: 2.118
CNN seed=1  test MAE: 2.155
CNN seed=2  test MAE: 2.109
CNN seed=3  test MAE: 2.126
CNN seed=4  test MAE: 2.160
CNN ortalama: 2.133  std: 0.020
